# ADL Results Explorer

Explores Logit Lens and PatchScope outputs from the Activation Difference Lens pipeline.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt

# --- Configuration (edit these) ---
RESULTS_DIR = Path(
    "../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/uprog_wide_parallel_api/activation_difference_lens"
)
LAYERS = [7, 14, 15]
DATASET = "tulu-3-sft-olmo-2-mixture"
LOGIT_LENS_POSITION = -1  # Position for per-position logit lens view
PATCHSCOPE_POSITION = -1  # Position for per-position patchscope view
N_POSITIONS = 128  # Total positions (config: n)
LOGIT_LENS_MAX_ROWS = 20  # Set to an integer to truncate logit lens tables
PATCHSCOPE_GRADER = "openai_gpt-5-mini"
MODEL_ID = "allenai/OLMo-2-0425-1B-DPO"

LAYER_DIRS = {layer: RESULTS_DIR / f"layer_{layer}" / DATASET for layer in LAYERS}

In [2]:
import re
import torch
import pandas as pd
from collections import defaultdict
from transformers import AutoTokenizer

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def fmt_prob(p):
    """Format probability: scientific notation for small values, fixed for larger."""
    if abs(p) < 0.01:
        return f"{p:.2e}"
    return f"{p:.4f}"


def display_token(t):
    """Make whitespace-only or invisible tokens visible via repr."""
    if not t.strip():
        return repr(t)
    return t


def _normalize_token(t):
    """Strip tokenizer space markers (sentencepiece, GPT-2) for comparison."""
    return t.replace("\u2581", "").replace("\u0120", "").strip()


def load_logit_lens(layer, pos, prefix=""):
    """Load logit lens .pt file. Returns (top_k_probs, top_k_indices, inv_probs, inv_indices)."""
    return torch.load(
        LAYER_DIRS[layer] / f"{prefix}logit_lens_pos_{pos}.pt", weights_only=True
    )


def decode_tokens(indices):
    return [tokenizer.decode([int(i)]) for i in indices]


def load_patchscope(layer, pos, prefix=""):
    """Load auto_patch_scope .pt file. Returns dict with tokens_at_best_scale, selected_tokens, etc."""
    return torch.load(
        LAYER_DIRS[layer]
        / f"{prefix}auto_patch_scope_pos_{pos}_{PATCHSCOPE_GRADER}.pt",
        weights_only=False,
    )


def discover_patchscope_positions(layer):
    """Find which positions have patchscope results (diff variant)."""
    positions = []
    for f in sorted(
        LAYER_DIRS[layer].glob(f"auto_patch_scope_pos_*_{PATCHSCOPE_GRADER}.pt")
    ):
        m = re.search(r"auto_patch_scope_pos_(\d+)_", f.name)
        if m:
            positions.append(int(m.group(1)))
    return positions


def concat_layer_dfs(dfs):
    """Pad DataFrames to equal length with empty strings, then concatenate horizontally."""
    max_len = max(len(df) for df in dfs)
    padded = []
    for df in dfs:
        if len(df) < max_len:
            pad = pd.DataFrame(
                {col: [""] * (max_len - len(df)) for col in df.columns},
                index=range(len(df), max_len),
            )
            df = pd.concat([df, pad], axis=0)
        padded.append(df)
    return pd.concat(padded, axis=1)


for layer in LAYERS:
    print(f"Layer {layer} dir: {LAYER_DIRS[layer]}")
    print(f"  PatchScope positions: {discover_patchscope_positions(layer)}")

Layer 7 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/uprog_wide_parallel_api/activation_difference_lens/layer_7/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 14 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/uprog_wide_parallel_api/activation_difference_lens/layer_14/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 15 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/uprog_wide_parallel_api/activation_difference_lens/layer_15/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]


## 1. Logit Lens Analysis

### 1A. Single Position

Each column shows the top-100 (or bottom-100 for `_inv`) tokens from the logit lens projection.  
Format: `token (softmax_prob)`

In [3]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {LOGIT_LENS_POSITION}:")
logit_lens_position_table(LOGIT_LENS_POSITION)

Logit lens at position -1:


layer_7                                                \
                       base             base_inv                       ft   
0           .Today (0.0250)      urrenc (0.0222)          .Today (0.0219)   
1          .Second (0.0111)       act (5.10e-03)         .Second (0.0103)   
2        Buccane (8.61e-03)       pos (4.94e-03)       Buccane (8.54e-03)   
3          /Area (6.32e-03)    askell (3.51e-03)         /Area (6.65e-03)   
4            .au (4.91e-03)      gons (3.30e-03)           .au (4.03e-03)   
5      /problems (3.83e-03)        دي (2.00e-03)     /problems (3.68e-03)   
6            aru (3.83e-03)        �� (2.00e-03)         /Math (3.34e-03)   
7      /entities (2.88e-03)      ejec (2.00e-03)         /bind (3.14e-03)   
8          /Math (2.72e-03)      azon (1.94e-03)     /entities (2.85e-03)   
9       /problem (2.64e-03)     fácil (1.82e-03)      /problem (2.78e-03)   
10      /respond (2.55e-03)      anth (1.76e-03)     /activity (2.69e-03)   
11         /bind (2.55e-03)     posix (1.71e-03)          oire (2.69e-03)   
12          fter (2.47e-03)  essional (1.66e-03)      /respond (2.44e-03)   
13    confidence (2.26e-03)  Optional (1.56e-03)   persistence (2.30e-03)   
14     /activity (2.18e-03)      Vers (1.51e-03)          fter (2.30e-03)   
15     /operator (2.18e-03)        vs (1.46e-03)    confidence (2.17e-03)   
16   persistence (2.12e-03)    Phones (1.46e-03)          ilot (2.17e-03)   
17          ilot (2.04e-03)  >Welcome (1.25e-03)        soever (2.17e-03)   
18     belonging (1.98e-03)      orst (1.25e-03)           aru (2.09e-03)   
19     .AddRange (1.87e-03)       med (1.25e-03)     /operator (2.09e-03)   

                                                                     \
                 ft_inv                   diff             diff_inv   
0       urrenc (0.0145)        gorith (0.0104)     ainer (7.75e-03)   
1        pos (5.31e-03)        onta (6.68e-03)   Passage (5.83e-03)   
2        act (4.85e-03)       encia (6.68e-03)   passing (5.68e-03)   
3     askell (4.85e-03)     trovare (5.52e-03)       buz (5.31e-03)   
4         �� (3.22e-03)   -scalable (4.30e-03)      htub (5.31e-03)   
5       gons (3.13e-03)          }? (4.06e-03)       muc (4.55e-03)   
6      fácil (2.84e-03)   acimiento (3.80e-03)     arten (3.89e-03)   
7         دي (2.15e-03)     Masters (3.36e-03)      Pass (3.77e-03)   
8       anth (2.15e-03)      stride (2.96e-03)   Passing (3.77e-03)   
9       azon (1.95e-03)    Accounts (2.96e-03)      prox (3.13e-03)   
10       med (1.95e-03)         sab (2.79e-03)      auen (3.04e-03)   
11  essional (1.89e-03)        .;\n (2.61e-03)     -town (2.85e-03)   
12        vs (1.67e-03)       abbix (2.46e-03)       ans (2.76e-03)   
13    Phones (1.62e-03)       guard (2.46e-03)       ush (2.59e-03)   
14      ejec (1.53e-03)        ucci (2.46e-03)       cre (2.44e-03)   
15  Optional (1.43e-03)  Executable (2.46e-03)    endale (2.37e-03)   
16         د (1.34e-03)      resden (2.30e-03)      PASS (2.37e-03)   
17     cambi (1.30e-03)          si (2.30e-03)       ubb (2.37e-03)   
18       div (1.22e-03)        atra (2.17e-03)         ư (2.37e-03)   
19      Vers (1.15e-03)        (SQL (2.17e-03)     alary (2.29e-03)   

                layer_14                                               \
                    base               base_inv                    ft   
0            To (0.9141)          zoek (0.8555)           To (0.9141)   
1           The (0.0454)      contador (0.1309)          The (0.0400)   
2           Let (0.0139)           메 (9.46e-03)           In (0.0157)   
3            In (0.0130)         иск (3.49e-03)          Let (0.0122)   
4         ### (3.72e-03)     Produto (2.12e-03)          A (3.49e-03)   
5           A (2.56e-03)           � (1.42e-05)        ### (3.30e-03)   
6         For (1.14e-03)     Detalle (9.78e-06)        For (1.88e-03)   
7          As (9.16e-04)      Resets (9.78e-06)         ** (1.46e-03)   
8           I (7.82e-04)        

In [4]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {5}:")
logit_lens_position_table(5)

Logit lens at position 5:


layer_7                                                \
                       base             base_inv                       ft   
0         /problem (0.0398)       .vn (7.72e-03)        /problem (0.0410)   
1        /entities (0.0273)       (us (5.00e-03)       /entities (0.0265)   
2        /problems (0.0177)      sagt (4.15e-03)       /problems (0.0182)   
3           .Today (0.0101)       ]]; (4.15e-03)       /global (8.85e-03)   
4        /global (8.91e-03)        że (3.65e-03)        .Today (8.61e-03)   
5        /manage (7.60e-03)    testim (3.02e-03)       /manage (6.71e-03)   
6           /job (7.60e-03)      -ves (2.84e-03)          /job (6.71e-03)   
7   /preferences (6.29e-03)       ($. (2.84e-03)       /layout (5.74e-03)   
8        /layout (5.74e-03)       ')" (2.84e-03)  /preferences (5.22e-03)   
9      /provider (5.07e-03)     zeigt (2.67e-03)     /provider (5.07e-03)   
10       /crypto (4.91e-03)     feliz (2.21e-03)       /crypto (4.91e-03)   
11   /connection (4.33e-03)     spons (2.21e-03)   /connection (4.33e-03)   
12      /logging (4.06e-03)     lesen (2.08e-03)      /logging (4.06e-03)   
13       /engine (3.94e-03)       (!! (1.95e-03)       /engine (3.81e-03)   
14    WHATSOEVER (3.94e-03)   kontrol (1.95e-03)  /environment (3.81e-03)   
15  /environment (3.81e-03)    spiele (1.72e-03)          /reg (3.71e-03)   
16          /reg (3.81e-03)      helf (1.72e-03)    WHATSOEVER (3.71e-03)   
17       /dialog (3.48e-03)     scrut (1.62e-03)      /effects (3.59e-03)   
18      /effects (3.37e-03)    mostra (1.52e-03)       /dialog (3.48e-03)   
19        .Round (3.37e-03)       )": (1.52e-03)       /entity (3.17e-03)   

                                                                     \
                 ft_inv                  diff              diff_inv   
0        .vn (8.36e-03)         ucci (0.0376)         ropa (0.0371)   
1        (us (4.49e-03)       anno (8.91e-03)      vidéo (5.34e-03)   
2        ]]; (4.21e-03)       Pert (7.87e-03)         ー� (5.34e-03)   
3       sagt (3.97e-03)       heim (7.39e-03)      atter (4.70e-03)   
4         że (3.97e-03)      annon (5.74e-03)        vod (4.58e-03)   
5      zeigt (3.08e-03)        lij (5.07e-03)  versation (4.30e-03)   
6       -ves (3.08e-03)       acea (4.76e-03)     .spark (4.30e-03)   
7        ($. (2.88e-03)   independ (3.49e-03)        zie (3.34e-03)   
8     testim (2.88e-03)      avras (3.49e-03)   narrowly (3.23e-03)   
9        ')" (2.72e-03)        bel (3.39e-03)       yles (2.94e-03)   
10       (!! (2.26e-03)      recon (3.17e-03)         ín (2.52e-03)   
11     feliz (2.26e-03)      Licht (2.72e-03)        cot (2.44e-03)   
12     lesen (2.26e-03)        apo (2.64e-03)       aved (2.37e-03)   
13     spons (2.26e-03)      ombok (2.32e-03)        ños (2.23e-03)   
14   kontrol (1.65e-03)         lj (2.32e-03)        får (2.03e-03)   
15      helf (1.65e-03)        boy (2.12e-03)       così (1.97e-03)   
16    spiele (1.55e-03)        tat (2.04e-03)       rose (1.85e-03)   
17       )": (1.55e-03)        rus (1.98e-03)       rend (1.85e-03)   
18     scrut (1.46e-03)       alus (1.93e-03)       acak (1.79e-03)   
19      -git (1.46e-03)         co (1.81e-03)      etten (1.79e-03)   

            layer_14                                           \
                base              base_inv                 ft   
0         , (0.5664)     contador (0.8398)         , (0.6172)   
1       and (0.1455)   karakter (7.26e-03)       and (0.1377)   
2       the (0.1387)         bö (7.26e-03)       the (0.1021)   
3        in (0.0596)    kontrol (7.26e-03)       ' ' (0.0564)   
4       ' ' (0.0552)         �� (5.65e-03)        in (0.0542)   
5         a (0.0133)         �� (5.65e-03)         a (0.0120)   
6       ( (3.69e-03)      KANJI (3.43e-03)       ( (4.70e-03)   
7      to (3.42e-03)       samt (3.43e-03)      to (2.99e-03)   
8      of (2.93e-03)      subur (3.43e-03)      of (2.66e-03)   
9      by (2.64e-03)     kosten (2.67e-03)     

### 1B. Aggregated Across All Positions

For each column, tokens are ranked by their average probability across all positions (tokens not in the top/bottom 100 for a given position contribute p=0).  
Format: `token (avg_prob)`

In [5]:
def logit_lens_aggregated_single(layer):
    agg = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        token_prob_sum = defaultdict(float)
        for pos in range(N_POSITIONS):
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
        token_avg = {t: s / N_POSITIONS for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        limit = LOGIT_LENS_MAX_ROWS if LOGIT_LENS_MAX_ROWS is not None else 100
        agg[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            for t in sorted_tokens[:limit]
        ]

    max_len = max(len(v) for v in agg.values())
    for k in agg:
        agg[k] += [""] * (max_len - len(agg[k]))
    return pd.DataFrame(agg)


def logit_lens_aggregated():
    dfs = []
    for layer in LAYERS:
        df = logit_lens_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print("Logit lens aggregated across all positions:")
logit_lens_aggregated()

Logit lens aggregated across all positions:


layer_7                                                \
                       base             base_inv                       ft   
0        /entities (0.0260)         .vn (0.0199)       /entities (0.0279)   
1         /problem (0.0140)   /Register (0.0114)        /problem (0.0150)   
2      /problems (9.26e-03)    testim (6.91e-03)     /problems (9.97e-03)   
3        /global (6.76e-03)      sagt (6.52e-03)       /global (6.90e-03)   
4   /environment (6.08e-03)     asign (5.22e-03)     /provider (6.30e-03)   
5      /provider (5.89e-03)       -ie (4.93e-03)  /environment (6.00e-03)   
6         .Today (5.88e-03)     zeigt (4.07e-03)   /connection (5.99e-03)   
7    /connection (5.69e-03)        że (3.93e-03)        .Today (5.45e-03)   
8        /manage (5.62e-03)      -ves (3.30e-03)     /customer (4.93e-03)   
9      /customer (4.80e-03)   personn (2.84e-03)       /manage (4.89e-03)   
10  /preferences (4.26e-03)         ť (2.84e-03)  /preferences (4.06e-03)   
11       /shared (3.36e-03)     probs (2.77e-03)       /shared (3.42e-03)   
12       /dialog (3.29e-03)      elig (2.55e-03)       /dialog (3.40e-03)   
13      /account (3.17e-03)    ):\n\n (2.39e-03)      /account (3.38e-03)   
14       /entity (3.15e-03)      roku (2.33e-03)      libertin (3.26e-03)   
15      libertin (3.01e-03)     lesen (2.23e-03)       /entity (3.19e-03)   
16       /layout (2.87e-03)       )": (2.22e-03)      /effects (3.05e-03)   
17          .Try (2.85e-03)  ,,,,,,,, (2.20e-03)       /layout (3.02e-03)   
18      /effects (2.71e-03)    spiele (2.12e-03)          .Try (2.91e-03)   
19        /legal (2.67e-03)       (us (2.09e-03)        /legal (2.67e-03)   

                                                                    \
                 ft_inv                  diff             diff_inv   
0          .vn (0.0216)         acea (0.0420)     nails (4.99e-03)   
1    /Register (0.0111)         ucci (0.0164)       row (4.41e-03)   
2     testim (6.69e-03)       anno (9.35e-03)      ropa (4.02e-03)   
3       sagt (6.55e-03)      Blond (6.86e-03)      emos (3.70e-03)   
4        -ie (5.02e-03)        lij (5.91e-03)     calle (3.53e-03)   
5      asign (4.95e-03)       heim (5.64e-03)       cho (3.44e-03)   
6      zeigt (4.50e-03)        ple (4.30e-03)      yles (3.04e-03)   
7         że (3.72e-03)     ervers (3.26e-03)   Bedford (2.82e-03)   
8       -ves (3.31e-03)     ighted (3.01e-03)       rem (2.40e-03)   
9          ť (3.25e-03)       ając (2.46e-03)        ー� (2.35e-03)   
10     probs (2.86e-03)       orda (2.43e-03)      nail (2.11e-03)   
11   personn (2.77e-03)      avras (2.39e-03)        Ps (1.94e-03)   
12      roku (2.35e-03)        Lah (2.30e-03)      ason (1.90e-03)   
13      elig (2.33e-03)     itives (2.05e-03)     venta (1.82e-03)   
14     lesen (2.29e-03)      Arial (1.91e-03)    olicit (1.77e-03)   
15  ,,,,,,,, (2.27e-03)      annon (1.76e-03)   lements (1.71e-03)   
16    ):\n\n (2.21e-03)   independ (1.76e-03)     ISSUE (1.64e-03)   
17       )": (2.21e-03)     aturas (1.68e-03)       arp (1.60e-03)   
18       esl (2.00e-03)   markdown (1.54e-03)     atter (1.54e-03)   
19    spiele (1.95e-03)      pelic (1.48e-03)         ś (1.49e-03)   

             layer_14                                           \
                 base              base_inv                 ft   
0          , (0.7985)     contador (0.9630)         , (0.8096)   
1        ' ' (0.1085)    kontrol (3.28e-03)       ' ' (0.1174)   
2        the (0.0435)   karakter (2.40e-03)       the (0.0329)   
3        and (0.0314)       rekl (1.58e-03)       and (0.0244)   
4       in (6.13e-03)         bö (1.43e-03)       ( (5.47e-03)   
5        ( (4.52e-03)       zoek (1.13e-03)      in (4.47e-03)   
6       's (2.69e-03)    Produto (9.35e-04)      's (2.14e-03)   
7        a (1.70e-03)     testim (9.10e-04)       a (1.21e-03)   
8       to (1.08e-03)     bilder (7.93e-04)      to (8.67e-04)   
9        . (6.78e-04)       dara (7.58e-04)       . (6.73e

## 2. PatchScope Analysis

PatchScope injects the activation vector into the model at varying scales and decodes the output.  
Unlike logit lens, there are no inverse variants -- only `base`, `ft`, and `diff`.  
Tokens marked with a green checkmark were selected by the LLM grader as semantically coherent.

### 2A. Single Position

Shows tokens at the best scale found by the auto patch scope search.  
Format: `token (prob)` with `\u2705` if in `selected_tokens`

In [6]:
PS_VARIANTS = [("base", "base_"), ("ft", "ft_"), ("diff", "")]


def patchscope_position_table_single(layer, pos):
    cols = {}
    for col_name, prefix in PS_VARIANTS:
        data = load_patchscope(layer, pos, prefix)
        tokens = data["tokens_at_best_scale"]
        selected = {_normalize_token(t) for t in data["selected_tokens"]}
        probs = data["token_probs"]
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})"
            + (" \u2705" if _normalize_token(t) in selected else "")
            for t, p in zip(tokens, probs)
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = patchscope_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"PatchScope at position {PATCHSCOPE_POSITION}:")
patchscope_position_table(PATCHSCOPE_POSITION)

PatchScope at position -1:


layer_7                              \
                       base                          ft   
0           .Today (0.0248)                 To (0.4235)   
1        .Second (8.57e-03)                The (0.2350)   
2        Buccane (5.77e-03)                 It (0.0601)   
3        /Area (5.72e-03) ✅               Your (0.0439)   
4            aru (5.53e-03)                 In (0.0279)   
5            .au (5.36e-03)               This (0.0240)   
6     confidence (2.93e-03)        Certainly (0.0190) ✅   
7           fter (2.90e-03)            Given (0.0162) ✅   
8    /problems (2.81e-03) ✅               When (0.0135)   
9           ilot (2.73e-03)                 As (0.0135)   
10       /Math (2.54e-03) ✅          Based (9.39e-03) ✅   
11   /entities (2.21e-03) ✅              Let (8.23e-03)   
12   /activity (1.99e-03) ✅           Sure (8.09e-03) ✅   
13    /problem (1.97e-03) ✅             Here (6.76e-03)   
14       /bind (1.89e-03) ✅  Understanding (5.62e-03) ✅   
15   persistence (1.85e-03)               If (4.67e-03)   
16     belonging (1.80e-03)              ### (3.46e-03)   
17   /operator (1.69e-03) ✅            Yes (3.13e-03) ✅   
18    /respond (1.67e-03) ✅              You (2.93e-03)   
19     .AddRange (1.51e-03)               `` (2.61e-03)   

                                          layer_14                          \
                        diff                  base                      ft   
0             ' ' (7.94e-03)           To (0.7422)             To (0.7904)   
1    coordinate (7.09e-03) ✅          ### (0.1138)             ** (0.0944)   
2           might (5.76e-03)           ** (0.0688)            ### (0.0623)   
3          '\n\n' (4.50e-03)        Let (0.0537) ✅            Let (0.0347)   
4      synchron (4.14e-03) ✅          The (0.0154)          The (9.97e-03)   
5           Mon (3.74e-03) ✅  Certainly (1.43e-03)  Certainly (2.62e-03) ✅   
6            Than (3.57e-03)       Sure (9.84e-04)       Sure (1.73e-03) ✅   
7           encia (3.57e-03)         In (6.75e-04)           In (8.16e-04)   
8          reader (3.56e-03)         ## (6.75e-04)           ## (5.39e-04)   
9         items (3.42e-03) ✅    First (2.34e-04) ✅         Here (3.71e-04)   
10       target (3.31e-03) ✅    Given (2.34e-04) ✅        Given (2.49e-04)   
11    financial (3.17e-03) ✅         We (1.25e-04)        First (2.07e-04)   
12          super (2.82e-03)        1 (1.23e-04) ✅         This (1.45e-04)   
13     deadline (2.69e-03) ✅    Alright (1.11e-04)          For (1.33e-04)   
14            ... (2.49e-03)     This (9.74e-05) ✅           As (1.25e-04)   
15      morning (2.49e-03) ✅     Here (9.16e-05) ✅            1 (1.02e-04)   
16           '  ' (2.31e-03)       #### (8.96e-05)    Alright (9.97e-05) ✅   
17          supra (2.30e-03)        ``` (8.27e-05)           We (7.28e-05)   
18             Is (2.26e-03)        For (6.58e-05)            A (7.12e-05)   
19              1 (2.20e-03)         As (6.29e-05)            I (6.83e-05)   

                                          layer_15                        \
                        diff                  base                    ft   
0     Publication (0.0104) ✅         To (0.4434) ✅           To (0.4785)   
1          '\x0c' (4.67e-03)           ** (0.2695)           ** (0.3730)   
2            Luck (3.86e-03)          ### (0.2373)          ### (0.1069)   
3            angu (2.69e-03)        Let (0.0250) ✅        Let (0.0145) ✅   
4    Additional (2.55e-03) ✅          The (0.0172)          The (0.0128)   
5             Imp (2.44e-03)  Certainly (2.32e-03)  Certainly (4.70e-03)   
6             azy (2.37e-03)       Sure (1.41e-03)       Sure (2.21e-03)   
7             ico (2.26e-03)         In (1.24e-03)         In (1.72e-03)   
8             agg (2.23e-03)         ## (7.55e-04)         ## (6.37e-04)   
9          BOOK (1.93e-03) ✅    Given (3.57e-04) ✅       Here (4.37e-04)   
10             Es (1.83e-03)    First (2.44e-04) ✅          A (4.37e-04)   
11      Harness 

### 2B. Aggregated Across All PatchScope Positions

Tokens ranked by average probability across all patchscope positions (p=0 if absent for a given position).  
Green checkmark if the token was in `selected_tokens` for **any** position.  
Format: `token (avg_prob)`

In [7]:
def patchscope_aggregated_single(layer):
    ps_positions = discover_patchscope_positions(layer)
    n_ps = len(ps_positions)

    cols = {}
    for col_name, prefix in PS_VARIANTS:
        token_prob_sum = defaultdict(float)
        ever_selected = set()
        for pos in ps_positions:
            data = load_patchscope(layer, pos, prefix)
            tokens = data["tokens_at_best_scale"]
            probs = data["token_probs"]
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
            ever_selected.update(_normalize_token(t) for t in data["selected_tokens"])

        token_avg = {t: s / n_ps for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            + (" \u2705" if _normalize_token(t) in ever_selected else "")
            for t in sorted_tokens
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_aggregated():
    dfs = []
    for layer in LAYERS:
        df = patchscope_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


ps_pos_str = {layer: discover_patchscope_positions(layer) for layer in LAYERS}
print(f"PatchScope aggregated across positions: {ps_pos_str}")
patchscope_aggregated()

PatchScope aggregated across positions: {7: [0, 1, 2, 3, 4, 5], 14: [0, 1, 2, 3, 4, 5], 15: [0, 1, 2, 3, 4, 5]}


layer_7                             \
                         base                         ft   
0                 's (0.0349)                -> (0.0614)   
1            solve (0.0318) ✅         problem (0.0468) ✅   
2                the (0.0295)                's (0.0447)   
3          problem (0.0284) ✅             :\n\n (0.0316)   
4         /problem (0.0268) ✅            '\n\n' (0.0237)   
5              :\n\n (0.0228)               you (0.0197)   
6                 -> (0.0198)        /problem (0.0191) ✅   
7        /entities (0.0175) ✅               the (0.0164)   
8        /problems (0.0141) ✅           solve (0.0143) ✅   
9                you (0.0123)       /entities (0.0121) ✅   
10       /manage (9.02e-03) ✅              '\n' (0.0117)   
11            this (8.73e-03)             :\n (9.61e-03)   
12             :\n (6.30e-03)     /problems (9.40e-03) ✅   
13          '\n\n' (6.25e-03)               , (9.35e-03)   
14            your (5.60e-03)     statement (7.93e-03) ✅   
15    understand (4.94e-03) ✅           seems (6.97e-03)   
16       address (4.50e-03) ✅            this (6.80e-03)   
17        .Today (4.35e-03) ✅      question (6.77e-03) ✅   
18  /preferences (3.71e-03) ✅    understand (5.95e-03) ✅   
19       /global (3.66e-03) ✅            your (5.51e-03)   
20        solves (3.26e-03) ✅              is (4.69e-03)   
21       analyze (3.07e-03) ✅              ’s (4.58e-03)   
22            '\n' (3.04e-03)      problems (4.57e-03) ✅   
23              ’s (2.96e-03)       address (3.84e-03) ✅   
24      question (2.66e-03) ✅               : (3.82e-03)   
25          math (2.61e-03) ✅       /manage (3.72e-03) ✅   
26          /job (2.50e-03) ✅        puzzle (3.35e-03) ✅   
27       /crypto (2.45e-03) ✅          math (3.28e-03) ✅   
28     /provider (2.38e-03) ✅        .Today (2.99e-03) ✅   
29      problems (2.34e-03) ✅       /global (2.85e-03) ✅   
30        puzzle (2.21e-03) ✅        solves (2.72e-03) ✅   
31       /layout (1.88e-03) ✅  /preferences (2.16e-03) ✅   
32      /logging (1.86e-03) ✅      solution (1.86e-03) ✅   
33           seems (1.85e-03)     /provider (1.86e-03) ✅   
34       /object (1.81e-03) ✅       /layout (1.71e-03) ✅   
35               , (1.81e-03)          /job (1.70e-03) ✅   
36   /connection (1.63e-03) ✅       example (1.64e-03) ✅   
37               : (1.58e-03)      /logging (1.61e-03) ✅   
38     /activity (1.52e-03) ✅       puzzles (1.54e-03) ✅   
39     statement (1.48e-03) ✅        tackle (1.52e-03) ✅   
40        tackle (1.43e-03) ✅          task (1.49e-03) ✅   
41       /engine (1.36e-03) ✅       /object (1.41e-03) ✅   
42       puzzles (1.25e-03) ✅       /engine (1.28e-03) ✅   
43              we (1.21e-03)       /crypto (1.27e-03) ✅   
44      /effects (1.20e-03) ✅      /effects (1.17e-03) ✅   
45          step (1.16e-03) ✅  /environment (1.16e-03) ✅   
46  /environment (1.16e-03) ✅     /activity (1.04e-03) ✅   
47       /shared (1.09e-03) ✅        prompt (1.02e-03) ✅   
48       /entity (1.04e-03) ✅       analyze (1.01e-03) ✅   
49  /application (1.02e-03) ✅        /legal (1.00e-03) ✅   
50        /legal (1.00e-03) ✅   /connection (9.88e-04) ✅   
51          begins (9.75e-04)         break (9.22e-04) ✅   
52        answer (8.95e-04) ✅      exercise (8.96e-04) ✅   
53      exercise (8.71e-04) ✅           .\n\n (8.69e-04)   
54          word (8.57e-04) ✅        answer (8.41e-04) ✅   
55          task (7.95e-04) ✅      requires (7.67e-04) ✅   
56        decode (7.76e-04) ✅            /con (7.29e-04)   
57             for (7.50e-04)          step (7.16e-04) ✅   
58        solved (7.16e-04) ✅             /pr (7.05e-04)   
59             /pr (6.92e-04)         appears (6.82e-04)   
60      /testing (6.51e-04) ✅          begins (6.56e-04)   
61          /reg (6.06e-04) ✅       clarify (6.08e-04) ✅   
62            /con (5.45e-04)      /testing (6.02e-04) ✅   
63        /tasks (5.34e-04) ✅          /reg (5.97e-04) ✅   
64          .Round (5.22e-04)  /application (5.55e-04) ✅   
65

## 3. Diff Logit Lens Across Positions

Shows only the **diff** variant of the logit lens for selected positions across all layers.
Format: `token (softmax_prob)`

In [8]:
DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3, 10, 50, 100]


def logit_lens_diff_positions_table():
    """Show diff logit lens across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in DIFF_POSITIONS:
            prefix, pi, ii = LL_VARIANTS["diff"]
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            col = [f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)]
            if LOGIT_LENS_MAX_ROWS is not None:
                col = col[:LOGIT_LENS_MAX_ROWS]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame(col_data)
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"Logit lens DIFF across positions: {DIFF_POSITIONS}")
logit_lens_diff_positions_table()

Logit lens DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_7                                                \
                  pos_-3                 pos_-1                  pos_0   
0          Sink (0.0286)        gorith (0.0104)         imo (9.77e-03)   
1    caratter (9.89e-03)        onta (6.68e-03)        agra (8.06e-03)   
2        kins (7.23e-03)       encia (6.68e-03)        rate (7.11e-03)   
3          el (7.23e-03)     trovare (5.52e-03)        ates (5.22e-03)   
4        pard (6.81e-03)   -scalable (4.30e-03)    ategorie (4.88e-03)   
5        elin (6.38e-03)          }? (4.06e-03)         ate (4.88e-03)   
6       orama (4.67e-03)   acimiento (3.80e-03)       avras (4.33e-03)   
7         apo (4.12e-03)     Masters (3.36e-03)        ucci (4.06e-03)   
8       elian (3.88e-03)      stride (2.96e-03)     Opening (3.37e-03)   
9         alu (3.63e-03)    Accounts (2.96e-03)        |,\n (3.16e-03)   
10   enslaved (3.20e-03)         sab (2.79e-03)        reib (3.16e-03)   
11  selection (3.02e-03)        .;\n (2.61e-03)         tat (2.98e-03)   
12        oll (2.66e-03)       abbix (2.46e-03)   slightest (2.98e-03)   
13        loi (2.66e-03)       guard (2.46e-03)    independ (2.98e-03)   
14         ån (2.66e-03)        ucci (2.46e-03)        bill (2.98e-03)   
15        Loc (2.66e-03)  Executable (2.46e-03)          Fe (2.98e-03)   
16          片 (2.21e-03)      resden (2.30e-03)     aproxim (2.98e-03)   
17      posto (2.21e-03)          si (2.30e-03)        meno (2.46e-03)   
18         но (2.08e-03)        atra (2.17e-03)        orsi (2.46e-03)   
19       otec (2.08e-03)        (SQL (2.17e-03)         iji (2.32e-03)   

                                                                          \
                    pos_1                 pos_2                    pos_3   
0          avras (0.0129)         heim (0.0155)            acea (0.0291)   
1         orsi (6.47e-03)         ucci (0.0121)            anno (0.0146)   
2      surplus (6.07e-03)         acea (0.0114)            ucci (0.0121)   
3          ate (6.07e-03)         anno (0.0100)          heim (6.90e-03)   
4         bags (5.71e-03)       lore (8.30e-03)      independ (6.47e-03)   
5        Trail (5.37e-03)      pelic (7.81e-03)         pelic (6.07e-03)   
6         ..." (4.30e-03)   independ (6.07e-03)          lore (5.04e-03)   
7         rate (4.30e-03)        tat (4.18e-03)           tat (4.46e-03)   
8       stride (4.30e-03)       rate (3.94e-03)            co (3.94e-03)   
9         ates (4.18e-03)    surplus (3.69e-03)         ativa (3.36e-03)   
10         apo (3.57e-03)      avras (3.69e-03)           apo (3.16e-03)   
11         owi (3.57e-03)       Pert (3.46e-03)          Pert (2.87e-03)   
12        ucci (3.46e-03)    isateur (2.53e-03)   recognition (2.78e-03)   
13        heim (3.25e-03)        ily (2.53e-03)            lj (2.70e-03)   
14          ta (3.14e-03)      annon (2.46e-03)      antanamo (2.70e-03)   
15    antanamo (2.87e-03)         lj (2.46e-03)           OCC (2.61e-03)   
16        Pert (2.78e-03)       ates (2.30e-03)           995 (2.38e-03)   
17       posed (2.70e-03)         Zu (2.24e-03)         ombok (1.85e-03)   
18        onta (2.61e-03)        lij (2.04e-03)           rus (1.85e-03)   
19   slightest (2.53e-03)        ate (2.04e-03)         indiv (1.74e-03)   

                                                                     \
                  pos_10               pos_50               pos_100   
0          anno (0.0498)        acea (0.0437)         acea (0.0515)   
1          ucci (0.0221)        ucci (0.0171)         ucci (0.0122)   
2          heim (0.0111)     Blond (8.12e-03)      Blond (8.97e-03)   
3        acea (5.62e-03)       lij (6.32e-03)        lij (6.13e-03)   
4         apo (4.09e-03)      anno (6.32e-03)       heim (4.49e-03)   
5        Pert (4.09e-03)      heim (5.55e-03)     ighted (4.21e-03)   
6       annon (3.85e-03)       ple (5.22e-03)       anno (3.97e-03)   
7         lij (3.40e-03)    ervers (4.91e-03)        ple (3.51e-03)   
8    

## 4. Diff PatchScope Across Positions

Shows only the **diff** variant of PatchScope for selected positions across all layers.
Format: `token (prob)` with `✅` if in `selected_tokens`

In [9]:
PS_DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3]


def patchscope_diff_positions_table():
    """Show diff patchscope across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in PS_DIFF_POSITIONS:
            try:
                data = load_patchscope(layer, pos, prefix="")
            except FileNotFoundError:
                col_data[f"pos_{pos}"] = ["(not available)"]
                continue
            tokens = data["tokens_at_best_scale"]
            selected = {_normalize_token(t) for t in data["selected_tokens"]}
            probs = data["token_probs"]
            col = [
                f"{display_token(t)} ({fmt_prob(p)})"
                + (" ✅" if _normalize_token(t) in selected else "")
                for t, p in zip(tokens, probs)
            ]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame({k: pd.Series(v) for k, v in col_data.items()}).fillna(
            ""
        )
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"PatchScope DIFF across positions: {DIFF_POSITIONS}")
patchscope_diff_positions_table()

PatchScope DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_7                            \
                        pos_-3                    pos_-1   
0              Edited (0.0171)            ' ' (7.94e-03)   
1                 alu (0.0145)   coordinate (7.09e-03) ✅   
2                ih (8.47e-03)          might (5.76e-03)   
3             atory (8.43e-03)         '\n\n' (4.50e-03)   
4              haft (5.96e-03)     synchron (4.14e-03) ✅   
5               que (4.55e-03)          Mon (3.74e-03) ✅   
6        statutes (4.18e-03) ✅           Than (3.57e-03)   
7              Sink (4.12e-03)          encia (3.57e-03)   
8            laws (4.10e-03) ✅         reader (3.56e-03)   
9                og (3.45e-03)        items (3.42e-03) ✅   
10              ate (3.23e-03)       target (3.31e-03) ✅   
11    Legislation (2.85e-03) ✅    financial (3.17e-03) ✅   
12        character (2.80e-03)          super (2.82e-03)   
13        selection (2.78e-03)     deadline (2.69e-03) ✅   
14             Hier (2.66e-03)            ... (2.49e-03)   
15             Meer (2.55e-03)      morning (2.49e-03) ✅   
16   AttributeError (2.52e-03)           '  ' (2.31e-03)   
17            Goose (2.42e-03)          supra (2.30e-03)   
18            lég (2.36e-03) ✅             Is (2.26e-03)   
19              AZE (2.16e-03)              1 (2.20e-03)   

                                                                          \
                    pos_0                 pos_1                    pos_2   
0           bear (0.0879)        dog (0.0483) ✅              -> (0.0485)   
1            ... (0.0434)          ' ' (0.0396)       chicken (0.0192) ✅   
2              " (0.0242)    chicken (0.0194) ✅      shrimp (8.13e-03) ✅   
3            man (0.0171)     person (0.0186) ✅       sauce (8.03e-03) ✅   
4          ' ' (9.11e-03)         '\n' (0.0179)             " (7.90e-03)   
5            ' (6.98e-03)            - (0.0144)           -ch (5.72e-03)   
6       family (6.79e-03)        man (0.0135) ✅        food (5.28e-03) ✅   
7           -> (5.87e-03)        bee (0.0126) ✅        gloves (5.17e-03)   
8          ... (5.14e-03)     turtle (0.0121) ✅           mer (4.98e-03)   
9          may (4.93e-03)        cat (0.0111) ✅          time (4.73e-03)   
10           ( (4.20e-03)           -> (0.0106)   chocolate (4.60e-03) ✅   
11           B (3.87e-03)     monkey (0.0104) ✅            ch (4.51e-03)   
12   strange (3.55e-03) ✅       blue (9.45e-03)             : (4.33e-03)   
13        call (3.20e-03)          : (8.87e-03)        fish (4.30e-03) ✅   
14     happy (3.15e-03) ✅       last (6.29e-03)        soup (4.24e-03) ✅   
15      fine (3.13e-03) ✅     '\n\n' (5.88e-03)     dinners (3.95e-03) ✅   
16      nice (2.95e-03) ✅    chick (4.63e-03) ✅      dishes (3.91e-03) ✅   
17      blue (2.53e-03) ✅   pigeon (4.47e-03) ✅             > (3.80e-03)   
18      kind (2.19e-03) ✅     dove (4.34e-03) ✅       honey (3.68e-03) ✅   
19        wish (2.16e-03)      guy (4.28e-03) ✅          '\n' (3.62e-03)   

                                          layer_14                            \
                    pos_3                   pos_-3                    pos_-1   
0             -> (0.0261)            sond (0.0195)    Publication (0.0104) ✅   
1              - (0.0178)            HEEL (0.0189)         '\x0c' (4.67e-03)   
2             -- (0.0110)         suger (8.70e-03)           Luck (3.86e-03)   
3        ice (9.80e-03) ✅           erb (6.76e-03)           angu (2.69e-03)   
4   -colored (8.16e-03) ✅           obs (6.36e-03)   Additional (2.55e-03) ✅   
5       blue (7.77e-03) ✅          inne (5.34e-03)            Imp (2.44e-03)   
6            : (7.75e-03)           opc (4.74e-03)            azy (2.37e-03)   
7      chicken (7.16e-03)        :]\n\n (4.67e-03)            ico (2.26e-03)   
8           is (6.97e-03)          kost (4.66e-03)            agg (2.23e-03)   
9            – (6.81e-03)    -animate (4.59e-03) ✅         BOOK (1.93e-03) ✅   
10         ' ' (6.32e-03)   -scalable (4.20e-03) ✅             Es (1.